# Project 5 — Advanced Solution

This notebook provides production-quality solutions for both project goals:

1. A **class-based context manager** that lazily reads CSV records as named tuples.
2. A **generator-based context manager** implemented with `contextlib.contextmanager`.

Both solutions:

- require only a file name/path;
- use `csv.reader`;
- detect the CSV dialect with `csv.Sniffer`;
- rewind the file after dialect detection;
- preserve every data value as a string;
- validate the header instead of silently changing field names;
- close the file reliably, including when exceptions occur;
- produce records lazily rather than loading the entire file into memory.


## Design decisions

- Files are opened with `newline=''`, as recommended by the `csv` module.
- `utf-8-sig` is used so a UTF-8 byte-order mark, when present, does not become part of the first field name.
- Only a bounded sample is read for dialect detection; the data rows remain lazy.
- Invalid, duplicated, blank, keyword, or underscore-prefixed headers raise a clear exception. This is safer than using `namedtuple(..., rename=True)`, which would silently change the CSV field names.
- Iterators are intended to be consumed **inside** their `with` block because the underlying file is closed when the context exits.


In [1]:
import csv
import keyword
import re
from collections import namedtuple
from contextlib import contextmanager
from itertools import islice
from pathlib import Path


_SNIFF_SAMPLE_SIZE = 8 * 1024


## Shared validation and parsing helpers

The two public solutions share these small helpers so behavior remains consistent and parsing logic is not duplicated.


In [2]:
class CSVNamedTupleError(Exception):
    """Base exception for this project."""


class CSVHeaderError(CSVNamedTupleError, ValueError):
    """Raised when a CSV header cannot be used as named-tuple fields."""


class CSVRowError(CSVNamedTupleError, ValueError):
    """Raised when a data row has an unexpected number of fields."""


_COMMON_DELIMITERS = ",;\t|:"


def _fallback_dialect(sample):
    """Build a conservative fallback dialect from the first non-empty line."""
    first_line = next(
        (line for line in sample.splitlines() if line.strip()),
        "",
    )

    # Prefer the delimiter occurring most often in the header.
    # Comma wins ties, which matches the normal CSV default.
    delimiter = max(
        _COMMON_DELIMITERS,
        key=lambda candidate: (first_line.count(candidate), candidate == ","),
    )

    if first_line.count(delimiter) == 0:
        delimiter = ","

    class FallbackDialect(csv.excel):
        pass

    FallbackDialect.delimiter = delimiter
    return FallbackDialect


def _sniff_dialect(file_obj):
    """Detect a usable CSV dialect from a bounded sample and rewind the file."""
    sample = file_obj.read(_SNIFF_SAMPLE_SIZE)
    file_obj.seek(0)

    if not sample:
        raise CSVHeaderError("The CSV file is empty; a header row is required.")

    try:
        dialect = csv.Sniffer().sniff(
            sample,
            delimiters=_COMMON_DELIMITERS,
        )
    except (csv.Error, TypeError, ValueError):
        dialect = _fallback_dialect(sample)

    delimiter = getattr(dialect, "delimiter", None)
    delimiter_is_valid = (
        isinstance(delimiter, str)
        and len(delimiter) == 1
        and delimiter not in "\r\n\x00"
    )

    if not delimiter_is_valid:
        dialect = _fallback_dialect(sample)

    # Validate the dialect at reader-construction time. Some Python versions
    # allow Sniffer to return a dialect that csv.reader later rejects.
    try:
        csv.reader([], dialect=dialect)
    except (csv.Error, TypeError, ValueError):
        dialect = _fallback_dialect(sample)
        csv.reader([], dialect=dialect)

    return dialect


def _validate_header(header, path):
    """Validate that header values are legal, unique named-tuple fields."""
    if not header:
        raise CSVHeaderError(
            "CSV file {!s} has an empty header row.".format(path)
        )

    errors = []

    blank_fields = [index + 1 for index, name in enumerate(header) if not name]
    if blank_fields:
        errors.append("blank field name(s) at position(s) {}".format(blank_fields))

    invalid_fields = [
        name
        for name in header
        if name
        and (
            not name.isidentifier()
            or keyword.iskeyword(name)
            or name.startswith("_")
        )
    ]
    if invalid_fields:
        errors.append(
            "invalid Python field name(s): {}".format(invalid_fields)
        )

    duplicates = sorted({name for name in header if header.count(name) > 1})
    if duplicates:
        errors.append("duplicate field name(s): {}".format(duplicates))

    if errors:
        raise CSVHeaderError(
            "Invalid header in {!s}: {}. ".format(path, "; ".join(errors))
            + "CSV headers must be unique valid Python identifiers because "
            + "they become named-tuple field names."
        )


def _row_type_name(path):
    """Create a valid, descriptive named-tuple type name from a file path."""
    stem = re.sub(r"\W+", "_", path.stem).strip("_")
    if not stem or stem[0].isdigit() or keyword.iskeyword(stem):
        stem = "CSV_{}".format(stem or "Data")
    return "{}Row".format(stem)


def _prepare_reader(file_obj, path):
    """Create the csv.reader, consume its header, and build the row type."""
    dialect = _sniff_dialect(file_obj)
    reader = csv.reader(file_obj, dialect=dialect)

    try:
        header = next(reader)
    except StopIteration:
        raise CSVHeaderError(
            "CSV file {!s} does not contain a header row.".format(path)
        )

    _validate_header(header, path)
    row_type = namedtuple(_row_type_name(path), header)
    return reader, row_type, tuple(header)


def _build_record(raw_row, row_type, fieldnames, line_number, path):
    """Validate one raw row and convert it to the generated named tuple."""
    expected = len(fieldnames)
    actual = len(raw_row)

    if actual != expected:
        raise CSVRowError(
            "Malformed row in {!s} near CSV line {}: expected {} fields, "
            "received {}. Row={!r}".format(
                path, line_number, expected, actual, raw_row
            )
        )

    return row_type(*raw_row)

# Goal 1 — Class-based context manager

`CSVNamedTupleReader` implements both the context-manager protocol and iterator protocol:

- `__enter__` opens and initializes the CSV reader.
- `__exit__` always closes the file.
- `__iter__` and `__next__` make the object a lazy one-pass iterator.
- `fieldnames` exposes the validated header while inside the context.


In [3]:
class CSVNamedTupleReader:
    """Lazily read CSV rows as named tuples inside a managed context."""

    def __init__(self, file_name):
        self._path = Path(file_name)
        self._file = None
        self._reader = None
        self._row_type = None
        self._fieldnames = None

    def __enter__(self):
        if self._file is not None:
            raise RuntimeError("This CSV reader is already inside a context.")

        self._file = self._path.open(
            mode="r",
            encoding="utf-8-sig",
            newline="",
        )

        try:
            (
                self._reader,
                self._row_type,
                self._fieldnames,
            ) = _prepare_reader(self._file, self._path)
        except Exception:
            self._file.close()
            self._file = None
            raise

        return self

    def __exit__(self, exc_type, exc_value, traceback):
        file_obj = self._file

        # Clear internal references even if close() itself were to fail.
        self._file = None
        self._reader = None
        self._row_type = None
        self._fieldnames = None

        if file_obj is not None:
            file_obj.close()

        # Returning False propagates exceptions raised inside the with block.
        return False

    def __iter__(self):
        return self

    def __next__(self):
        if self._reader is None:
            raise RuntimeError(
                "CSVNamedTupleReader must be consumed inside a with block."
            )

        raw_row = next(self._reader)
        return _build_record(
            raw_row=raw_row,
            row_type=self._row_type,
            fieldnames=self._fieldnames,
            line_number=self._reader.line_num,
            path=self._path,
        )

    @property
    def fieldnames(self):
        if self._fieldnames is None:
            raise RuntimeError("fieldnames is available only inside the context.")
        return self._fieldnames

    @property
    def row_type(self):
        if self._row_type is None:
            raise RuntimeError("row_type is available only inside the context.")
        return self._row_type

    @property
    def is_open(self):
        return self._file is not None and not self._file.closed


## Goal 1 usage

`islice` consumes only the requested rows, demonstrating lazy iteration. The examples are guarded so the notebook still runs when the project CSV files are not in the current directory.


In [4]:
cars_path = Path("cars.csv")

if cars_path.exists():
    with CSVNamedTupleReader(cars_path) as cars:
        print("Fields:", cars.fieldnames)
        print("Generated type:", cars.row_type)
        print()

        for car in islice(cars, 5):
            print(car)
            print("Car name:", car.Car)
            print("Origin:", car.Origin)
            print()
else:
    print("cars.csv was not found in", Path.cwd())


Fields: ('Car', 'MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Model', 'Origin')
Generated type: <class '__main__.carsRow'>

carsRow(Car='Chevrolet Chevelle Malibu', MPG='18.0', Cylinders='8', Displacement='307.0', Horsepower='130.0', Weight='3504.', Acceleration='12.0', Model='70', Origin='US')
Car name: Chevrolet Chevelle Malibu
Origin: US

carsRow(Car='Buick Skylark 320', MPG='15.0', Cylinders='8', Displacement='350.0', Horsepower='165.0', Weight='3693.', Acceleration='11.5', Model='70', Origin='US')
Car name: Buick Skylark 320
Origin: US

carsRow(Car='Plymouth Satellite', MPG='18.0', Cylinders='8', Displacement='318.0', Horsepower='150.0', Weight='3436.', Acceleration='11.0', Model='70', Origin='US')
Car name: Plymouth Satellite
Origin: US

carsRow(Car='AMC Rebel SST', MPG='16.0', Cylinders='8', Displacement='304.0', Horsepower='150.0', Weight='3433.', Acceleration='12.0', Model='70', Origin='US')
Car name: AMC Rebel SST
Origin: US

carsRow(Car='Ford T

# Goal 2 — Generator function with `contextmanager`

The decorated function manages the file resource, while the nested generator lazily converts each raw CSV row into the generated named-tuple type.


In [5]:
@contextmanager
def csv_namedtuple_reader(file_name):
    """Yield a lazy iterator of named tuples for the supplied CSV file."""
    path = Path(file_name)
    file_obj = path.open(
        mode="r",
        encoding="utf-8-sig",
        newline="",
    )

    try:
        reader, row_type, fieldnames = _prepare_reader(file_obj, path)

        def records():
            for raw_row in reader:
                yield _build_record(
                    raw_row=raw_row,
                    row_type=row_type,
                    fieldnames=fieldnames,
                    line_number=reader.line_num,
                    path=path,
                )

        yield records()
    finally:
        file_obj.close()


## Goal 2 usage


In [6]:
personal_info_path = Path("personal_info.csv")

if personal_info_path.exists():
    with csv_namedtuple_reader(personal_info_path) as people:
        for person in islice(people, 5):
            print(person)
            print("Name: {} {}".format(person.first_name, person.last_name))
            print("Language:", person.language)
            print()
else:
    print("personal_info.csv was not found in", Path.cwd())


personal_infoRow(ssn='100-53-9824', first_name='Sebastiano', last_name='Tester', gender='Male', language='Icelandic')
Name: Sebastiano Tester
Language: Icelandic

personal_infoRow(ssn='101-71-4702', first_name='Cayla', last_name='MacDonagh', gender='Female', language='Lao')
Name: Cayla MacDonagh
Language: Lao

personal_infoRow(ssn='101-84-0356', first_name='Nomi', last_name='Lipprose', gender='Female', language='Yiddish')
Name: Nomi Lipprose
Language: Yiddish

personal_infoRow(ssn='104-22-0928', first_name='Justinian', last_name='Kunzelmann', gender='Male', language='Dhivehi')
Name: Justinian Kunzelmann
Language: Dhivehi

personal_infoRow(ssn='104-84-7144', first_name='Claudianus', last_name='Brixey', gender='Male', language='Afrikaans')
Name: Claudianus Brixey
Language: Afrikaans



# Automated tests

These tests use temporary CSV files, so they do not depend on `cars.csv` or `personal_info.csv`. They verify delimiter detection, named-tuple fields, string values, laziness-compatible iteration, cleanup, malformed rows, and invalid headers.


In [7]:
from tempfile import TemporaryDirectory


def _write_text(path, text):
    path.write_text(text, encoding="utf-8")


with TemporaryDirectory() as temp_dir_name:
    temp_dir = Path(temp_dir_name)

    semicolon_file = temp_dir / "cars_sample.csv"
    _write_text(
        semicolon_file,
        "Car;MPG;Origin\n"
        "Chevrolet Chevelle Malibu;18.0;US\n"
        "Toyota Corolla;31.0;Japan\n",
    )

    comma_file = temp_dir / "people_sample.csv"
    _write_text(
        comma_file,
        "ssn,first_name,last_name,language\n"
        "100-53-9824,Sebastiano,Tester,Icelandic\n"
        "101-71-4702,Cayla,MacDonagh,Lao\n",
    )

    # Goal 1: class-based manager.
    class_reader = CSVNamedTupleReader(semicolon_file)
    assert not class_reader.is_open

    with class_reader as rows:
        assert rows.is_open
        assert iter(rows) is rows
        assert rows.fieldnames == ("Car", "MPG", "Origin")

        first = next(rows)
        second = next(rows)

        assert type(first)._fields == ("Car", "MPG", "Origin")
        assert first.Car == "Chevrolet Chevelle Malibu"
        assert first.MPG == "18.0"
        assert first.Origin == "US"
        assert all(isinstance(value, str) for value in first)
        assert second.Car == "Toyota Corolla"

        try:
            next(rows)
        except StopIteration:
            pass
        else:
            raise AssertionError("Iterator should be exhausted.")

    assert not class_reader.is_open

    # Goal 2: generator-based manager.
    with csv_namedtuple_reader(comma_file) as rows:
        first_two = list(islice(rows, 2))

    assert len(first_two) == 2
    assert type(first_two[0])._fields == (
        "ssn",
        "first_name",
        "last_name",
        "language",
    )
    assert first_two[0].first_name == "Sebastiano"
    assert first_two[1].language == "Lao"
    assert all(
        isinstance(value, str)
        for record in first_two
        for value in record
    )

    # A malformed row must fail with a useful error.
    malformed_file = temp_dir / "malformed.csv"
    _write_text(malformed_file, "a,b,c\n1,2\n")

    try:
        with CSVNamedTupleReader(malformed_file) as rows:
            next(rows)
    except CSVRowError as exc:
        assert "expected 3 fields" in str(exc)
    else:
        raise AssertionError("Malformed rows should raise CSVRowError.")

    # Invalid named-tuple headers must not be silently renamed.
    invalid_header_file = temp_dir / "invalid_header.csv"
    _write_text(invalid_header_file, "first name,class,value\nA,B,C\n")

    try:
        with csv_namedtuple_reader(invalid_header_file) as rows:
            next(rows)
    except CSVHeaderError as exc:
        assert "invalid Python field name" in str(exc)
    else:
        raise AssertionError("Invalid headers should raise CSVHeaderError.")

print("All automated tests passed.")


All automated tests passed.


# Value-added extensions

The following utilities are optional additions. They do not change either required solution or the rule that the CSV readers return strings.

They add:

- lazy filtering, projection, and explicit type conversion;
- tolerant ingestion with a bounded audit report;
- sequential processing of multiple compatible CSV files;
- streaming JSON Lines export;
- additional automated tests for the extended behavior.

## Extension 1 — Composable lazy processing

It is usually better for a CSV reader to focus only on parsing and resource management. Type conversion and business rules can then be composed as separate lazy iterator stages.

The functions below preserve streaming behavior: no full list is created unless the caller explicitly requests one.

In [8]:
def optional_float(value):
    """Convert a CSV string to float, treating common missing markers as None."""
    normalized = value.strip()
    if normalized in {"", "?", "NA", "N/A", "null", "None"}:
        return None
    return float(normalized)


def convert_record(record, converters):
    """Return one named tuple with selected fields explicitly converted."""
    converters = dict(converters)
    unknown_fields = sorted(set(converters).difference(record._fields))

    if unknown_fields:
        raise KeyError(
            "Converter supplied for unknown field(s): {}".format(unknown_fields)
        )

    replacements = {}

    for field_name, converter in converters.items():
        raw_value = getattr(record, field_name)

        try:
            replacements[field_name] = converter(raw_value)
        except (TypeError, ValueError, OverflowError) as exc:
            raise ValueError(
                "Could not convert field {!r} with value {!r} using {}.".format(
                    field_name,
                    raw_value,
                    getattr(converter, "__name__", repr(converter)),
                )
            ) from exc

    return record._replace(**replacements)


def convert_records(records, converters):
    """Lazily apply field converters to an iterable of named tuples."""
    converters = dict(converters)

    for record in records:
        yield convert_record(record, converters)


def filter_records(records, predicate):
    """Lazily yield only records for which predicate(record) is truthy."""
    for record in records:
        if predicate(record):
            yield record


def project_records(records, *field_names, **options):
    """Lazily select fields into a smaller named-tuple projection."""
    if not field_names:
        raise ValueError("At least one projected field is required.")

    type_name = options.pop("type_name", "CSVProjection")
    if options:
        raise TypeError(
            "Unexpected option(s): {}".format(sorted(options))
        )

    projection_type = namedtuple(type_name, field_names)

    for record in records:
        missing_fields = [
            field_name
            for field_name in field_names
            if field_name not in record._fields
        ]

        if missing_fields:
            raise KeyError(
                "Cannot project missing field(s): {}".format(missing_fields)
            )

        yield projection_type(
            *(getattr(record, field_name) for field_name in field_names)
        )

### Lazy pipeline example

The reader still emits strings. Conversion occurs only in the explicit `convert_records` stage, making the data-type policy visible and testable.

In [9]:
if cars_path.exists():
    with CSVNamedTupleReader(cars_path) as rows:
        converted = convert_records(
            rows,
            {
                "MPG": optional_float,
                "Horsepower": optional_float,
                "Model": int,
            },
        )

        efficient_cars = filter_records(
            converted,
            lambda car: car.MPG is not None and car.MPG >= 30,
        )

        preview = project_records(
            efficient_cars,
            "Car",
            "MPG",
            "Model",
            "Origin",
            type_name="EfficientCar",
        )

        for record in islice(preview, 10):
            print(record)
else:
    print("Pipeline example skipped because cars.csv was not found.")

EfficientCar(Car='Peugeot 304', MPG=30.0, Model=71, Origin='Europe')
EfficientCar(Car='Fiat 124B', MPG=30.0, Model=71, Origin='Europe')
EfficientCar(Car='Toyota Corolla 1200', MPG=31.0, Model=71, Origin='Japan')
EfficientCar(Car='Datsun 1200', MPG=35.0, Model=71, Origin='Japan')
EfficientCar(Car='Datsun B210', MPG=31.0, Model=74, Origin='Japan')
EfficientCar(Car='Toyota Corolla 1200', MPG=32.0, Model=74, Origin='Japan')
EfficientCar(Car='Toyota Corolla', MPG=31.0, Model=74, Origin='Japan')
EfficientCar(Car='Datsun 710', MPG=32.0, Model=74, Origin='Japan')
EfficientCar(Car='Fiat x1.9', MPG=31.0, Model=74, Origin='Europe')
EfficientCar(Car='Honda Civic CVCC', MPG=33.0, Model=75, Origin='Japan')


## Extension 2 — Tolerant reader with an audit report

Strict parsing is the safest default and remains the behavior of the two project solutions. In ingestion or exploratory workflows, however, it can be useful to skip malformed rows while recording exactly what was rejected.

The tolerant reader below:

- retains valid records lazily;
- counts rows as they are consumed;
- stores only a bounded number of issues;
- reports whether the iterator was fully consumed;
- still closes the file reliably.

In [10]:
CSVReadIssue = namedtuple(
    "CSVReadIssue",
    [
        "path",
        "line_number",
        "expected_fields",
        "actual_fields",
        "row",
    ],
)


class CSVReadReport:
    """Mutable audit information populated while a tolerant iterator runs."""

    def __init__(self, path, fieldnames, max_issues):
        self.path = Path(path)
        self.fieldnames = tuple(fieldnames)
        self.max_issues = max_issues
        self.rows_seen = 0
        self.rows_yielded = 0
        self.bad_rows = 0
        self.issues = []
        self.completed = False
        self.closed = False

    @property
    def omitted_issue_count(self):
        return self.bad_rows - len(self.issues)

    def to_dict(self):
        return {
            "path": str(self.path),
            "fieldnames": list(self.fieldnames),
            "rows_seen": self.rows_seen,
            "rows_yielded": self.rows_yielded,
            "bad_rows": self.bad_rows,
            "stored_issues": len(self.issues),
            "omitted_issue_count": self.omitted_issue_count,
            "completed": self.completed,
            "closed": self.closed,
        }

    def __repr__(self):
        return (
            "CSVReadReport(rows_seen={!r}, rows_yielded={!r}, "
            "bad_rows={!r}, completed={!r}, closed={!r})"
        ).format(
            self.rows_seen,
            self.rows_yielded,
            self.bad_rows,
            self.completed,
            self.closed,
        )


@contextmanager
def tolerant_csv_namedtuple_reader(file_name, max_issues=100):
    """
    Yield (records, report), skipping malformed-width rows.

    Only the first max_issues problems are retained to keep memory bounded.
    """
    if isinstance(max_issues, bool) or not isinstance(max_issues, int):
        raise TypeError("max_issues must be a non-negative integer.")
    if max_issues < 0:
        raise ValueError("max_issues must be non-negative.")

    path = Path(file_name)
    file_obj = path.open(
        mode="r",
        encoding="utf-8-sig",
        newline="",
    )

    iterator = None
    report = None

    try:
        reader, row_type, fieldnames = _prepare_reader(file_obj, path)
        report = CSVReadReport(path, fieldnames, max_issues)

        def records():
            expected = len(fieldnames)

            for raw_row in reader:
                report.rows_seen += 1
                actual = len(raw_row)

                if actual != expected:
                    report.bad_rows += 1

                    if len(report.issues) < max_issues:
                        report.issues.append(
                            CSVReadIssue(
                                path=path,
                                line_number=reader.line_num,
                                expected_fields=expected,
                                actual_fields=actual,
                                row=tuple(raw_row),
                            )
                        )
                    continue

                report.rows_yielded += 1
                yield row_type(*raw_row)

            report.completed = True

        iterator = records()

        try:
            yield iterator, report
        finally:
            iterator.close()
    finally:
        file_obj.close()
        if report is not None:
            report.closed = True

In [11]:
with TemporaryDirectory() as temp_dir_name:
    sample_path = Path(temp_dir_name) / "mixed_quality.csv"
    _write_text(
        sample_path,
        "id,name,score\n"
        "1,Ada,98\n"
        "2,Grace\n"
        "3,Linus,91\n"
        "4,Too,Many,Fields\n",
    )

    with tolerant_csv_namedtuple_reader(sample_path) as (records, report):
        valid_records = list(records)

    print("Valid records:")
    for record in valid_records:
        print(" ", record)

    print("\nAudit report:", report.to_dict())
    print("Recorded issues:")
    for issue in report.issues:
        print(" ", issue)

Valid records:
  mixed_qualityRow(id='1', name='Ada', score='98')
  mixed_qualityRow(id='3', name='Linus', score='91')

Audit report: {'path': 'C:\\Users\\user1\\AppData\\Local\\Temp\\tmp5kwgfujl\\mixed_quality.csv', 'fieldnames': ['id', 'name', 'score'], 'rows_seen': 4, 'rows_yielded': 2, 'bad_rows': 2, 'stored_issues': 2, 'omitted_issue_count': 0, 'completed': True, 'closed': True}
Recorded issues:
  CSVReadIssue(path=WindowsPath('C:/Users/user1/AppData/Local/Temp/tmp5kwgfujl/mixed_quality.csv'), line_number=3, expected_fields=3, actual_fields=2, row=('2', 'Grace'))
  CSVReadIssue(path=WindowsPath('C:/Users/user1/AppData/Local/Temp/tmp5kwgfujl/mixed_quality.csv'), line_number=5, expected_fields=3, actual_fields=4, row=('4', 'Too', 'Many', 'Fields'))


## Extension 3 — Sequential multi-file processing

`chain_csv_namedtuple_readers` processes compatible files as one lazy stream while opening only one file at a time.

Each yielded object includes:

- `source`: the originating `Path`;
- `record`: a named tuple using one common record type.

A schema mismatch raises `CSVHeaderError` as soon as the incompatible file is reached.

In [12]:
SourcedRecord = namedtuple("SourcedRecord", ["source", "record"])


@contextmanager
def chain_csv_namedtuple_readers(file_names):
    """Yield one lazy, schema-checked stream across multiple CSV files."""
    paths = tuple(Path(file_name) for file_name in file_names)

    if not paths:
        raise ValueError("At least one CSV file is required.")

    def records():
        expected_fieldnames = None
        common_row_type = None

        for path in paths:
            with CSVNamedTupleReader(path) as reader:
                if expected_fieldnames is None:
                    expected_fieldnames = reader.fieldnames
                    common_row_type = reader.row_type
                elif reader.fieldnames != expected_fieldnames:
                    raise CSVHeaderError(
                        "Schema mismatch in {!s}: expected {!r}, received {!r}.".format(
                            path,
                            expected_fieldnames,
                            reader.fieldnames,
                        )
                    )

                for record in reader:
                    # Normalize records from all files to one common tuple type.
                    yield SourcedRecord(
                        source=path,
                        record=common_row_type(*record),
                    )

    iterator = records()

    try:
        yield iterator
    finally:
        iterator.close()

## Extension 4 — Streaming JSON Lines export

JSON Lines stores one JSON object per line and is convenient for logs, data pipelines, and incremental processing. The exporter consumes records one at a time, so memory usage remains bounded.

In [13]:
import json


def write_json_lines(records, destination, encoding="utf-8"):
    """Stream named-tuple records to a JSON Lines file and return the count."""
    destination = Path(destination)
    count = 0

    with destination.open(
        mode="w",
        encoding=encoding,
        newline="",
    ) as file_obj:
        for record in records:
            if not hasattr(record, "_asdict"):
                raise TypeError(
                    "JSON Lines export requires named-tuple-like records."
                )

            json.dump(
                record._asdict(),
                file_obj,
                ensure_ascii=False,
            )
            file_obj.write("\n")
            count += 1

    return count

## Extended automated tests

These tests cover lazy conversion, error context, tolerant parsing, bounded issue storage, multi-file schema checking, source tracking, and streaming export.

In [14]:
with TemporaryDirectory() as temp_dir_name:
    temp_dir = Path(temp_dir_name)

    first_file = temp_dir / "part_1.csv"
    second_file = temp_dir / "part_2.csv"
    incompatible_file = temp_dir / "incompatible.csv"
    tolerant_file = temp_dir / "tolerant.csv"
    jsonl_file = temp_dir / "records.jsonl"

    _write_text(
        first_file,
        "id,name,score\n"
        "1,Ada,98.5\n"
        "2,Grace,99.0\n",
    )
    _write_text(
        second_file,
        "id,name,score\n"
        "3,Linus,91.5\n",
    )
    _write_text(
        incompatible_file,
        "id,full_name,score\n"
        "4,Guido,97.0\n",
    )
    _write_text(
        tolerant_file,
        "id,name,score\n"
        "1,Ada,98.5\n"
        "2,MissingScore\n"
        "3,Linus,91.5\n"
        "4,Too,Many,Columns\n",
    )

    # Lazy conversion, filtering, and projection.
    with CSVNamedTupleReader(first_file) as rows:
        pipeline = project_records(
            filter_records(
                convert_records(rows, {"id": int, "score": float}),
                lambda row: row.score >= 99,
            ),
            "id",
            "name",
            type_name="TopStudent",
        )
        top_students = list(pipeline)

    assert top_students == [top_students[0]]
    assert top_students[0].id == 2
    assert top_students[0].name == "Grace"
    assert isinstance(top_students[0].id, int)

    # Conversion errors include the field and original value.
    ConversionTestRow = namedtuple("ConversionTestRow", ["value"])
    try:
        convert_record(ConversionTestRow("not-a-number"), {"value": int})
    except ValueError as exc:
        message = str(exc)
        assert "value" in message
        assert "not-a-number" in message
    else:
        raise AssertionError("Invalid conversion should raise ValueError.")

    # Tolerant parsing and bounded issue collection.
    with tolerant_csv_namedtuple_reader(
        tolerant_file,
        max_issues=1,
    ) as (rows, report):
        valid_rows = list(rows)

    assert [row.id for row in valid_rows] == ["1", "3"]
    assert report.rows_seen == 4
    assert report.rows_yielded == 2
    assert report.bad_rows == 2
    assert len(report.issues) == 1
    assert report.omitted_issue_count == 1
    assert report.completed is True
    assert report.closed is True

    # Compatible files are chained lazily with source information.
    with chain_csv_namedtuple_readers(
        [first_file, second_file]
    ) as sourced_rows:
        combined = list(sourced_rows)

    assert [item.record.id for item in combined] == ["1", "2", "3"]
    assert combined[0].source == first_file
    assert combined[-1].source == second_file
    assert type(combined[0].record) is type(combined[-1].record)

    # An incompatible schema is detected when that file is reached.
    try:
        with chain_csv_namedtuple_readers(
            [first_file, incompatible_file]
        ) as sourced_rows:
            list(sourced_rows)
    except CSVHeaderError as exc:
        assert "Schema mismatch" in str(exc)
    else:
        raise AssertionError("Incompatible schemas should raise CSVHeaderError.")

    # JSON Lines export writes one object per record.
    with CSVNamedTupleReader(first_file) as rows:
        exported_count = write_json_lines(rows, jsonl_file)

    exported_objects = [
        json.loads(line)
        for line in jsonl_file.read_text(encoding="utf-8").splitlines()
    ]

    assert exported_count == 2
    assert exported_objects[0] == {
        "id": "1",
        "name": "Ada",
        "score": "98.5",
    }
    assert exported_objects[1]["name"] == "Grace"

print("All extended automated tests passed.")

All extended automated tests passed.


# Complexity, guarantees, and practical guidance

Let `n` be the number of consumed data rows and `w` the maximum row width.

## Core readers

- **Time:** `O(n × w)` to consume `n` rows.
- **Additional memory:** `O(w)` per current record, plus the bounded dialect sample.
- **Laziness:** records are created only when requested.
- **Resource safety:** files close through `__exit__` or `finally`, including exceptional paths.
- **One-pass behavior:** each returned iterator is consumable once.
- **Context lifetime:** consume the iterator inside its `with` block.

## Extensions

- Conversion, filtering, and projection remain lazy and use constant additional streaming memory.
- The tolerant reader stores at most `max_issues` detailed problems.
- Multi-file chaining opens only one input file at a time.
- JSON Lines export writes records incrementally rather than constructing one large JSON array.

## Choosing an API

Use the class-based solution when you want metadata such as `fieldnames`, `row_type`, or `is_open`.

```python
with CSVNamedTupleReader("cars.csv") as rows:
    print(rows.fieldnames)
    first_car = next(rows)
```

Use the decorated generator solution when you want the smallest public interface.

```python
with csv_namedtuple_reader("personal_info.csv") as rows:
    first_person = next(rows)
```

Use the tolerant reader only when skipping malformed-width rows is an explicit requirement.

```python
with tolerant_csv_namedtuple_reader("input.csv") as (rows, report):
    for row in rows:
        process(row)

print(report.to_dict())
```

## Important limitations

- `csv.Sniffer` is heuristic. The implementation validates its result and uses a conservative fallback, but highly unusual dialects may require an explicit custom reader in a real production system.
- Named-tuple fields must be unique valid Python identifiers.
- The strict readers intentionally stop at the first malformed-width row.
- All core reader values are strings. Optional type conversion is a separate downstream operation.